Make sure fights_cleaned.csv is updated by running data_cleaning.ipynb if changes are made

In [21]:
import pandas as pd
from pathlib import Path

# Set paths
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

events_path = DATA_RAW / 'ufc_events.csv'
fights_path = DATA_PROCESSED / 'fights_cleaned.csv'

(events_path, fights_path)

(WindowsPath('C:/Users/maxle/Documents/ML-Project-clone/data/raw/ufc_events.csv'),
 WindowsPath('C:/Users/maxle/Documents/ML-Project-clone/data/processed/fights_cleaned.csv'))

In [22]:
# Load raw data

events = pd.read_csv(events_path)
fights = pd.read_csv(fights_path)

print(events.columns)
print(fights.columns)

Index(['event_id', 'event_name', 'event_date', 'url_link'], dtype='str')
Index(['f1_id', 'f2_id', 'f1_name', 'f2_name', 'weight_class', 'f1_age_during',
       'f2_age_during', 'f1_height_cm', 'f2_height_cm', 'f1_knockdowns',
       'f2_knockdowns', 'f1_sig_strike_atts', 'f2_sig_strike_atts',
       'f1_sig_strikes', 'f2_sig_strikes', 'f1_tot_strike_atts',
       'f2_tot_strike_atts', 'f1_tot_strikes', 'f2_tot_strikes',
       'f1_takedown_atts', 'f2_takedown_atts', 'f1_takedowns', 'f2_takedowns',
       'f1_submissions', 'f2_submissions', 'f1_reversals', 'f2_reversals',
       'f1_ctrl_time', 'f2_ctrl_time', 'f1_head_strike_atts',
       'f2_head_strike_atts', 'f1_head_strikes', 'f2_head_strikes',
       'f1_body_strike_atts', 'f2_body_strike_atts', 'f1_body_strikes',
       'f2_body_strikes', 'f1_leg_strike_atts', 'f2_leg_strike_atts',
       'f1_leg_strikes', 'f2_leg_strikes', 'f1_dist_strike_atts',
       'f2_dist_strike_atts', 'f1_dist_strikes', 'f2_dist_strikes',
       'f1_clinc

In [23]:
# Join fights to events on event URL so each fight has its event date

# In events: URL column is `url_link`
# In fights: URL column is `event_url`

# This merge keeps **all** columns from both tables
fights_with_event = fights.merge(
    events,
    left_on='event_url',
    right_on='url_link',
    how='left',
)

# Just show a few columns for a quick visual check
fights_with_event[['f1_name', 'f2_name', 'event_name', 'event_date']].head()

,f1_name,f2_name,event_name,event_date
0,Cory Sandhagen,Umar Nurmagomedov,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
1,Shara Magomedov,Michal Oleksiejczuk,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
2,Jai Herbert,Rolando Bedoya,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
3,Azamat Murzakanov,Alonzo Menifield,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
4,Sedriques Dumas,Denis Tiuliulin,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03


In [24]:
# Prepare joined data (in memory only; we do not save fights_with_event.csv)

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Shared list of stat columns for cumulative stats and averages (used in later cells)
STAT_COLS = [
    'knockdowns', 'sig_strike_atts', 'sig_strikes', 'tot_strike_atts', 'tot_strikes',
    'takedown_atts', 'takedowns', 'submissions', 'reversals', 'ctrl_time',
    'head_strike_atts', 'head_strikes', 'body_strike_atts', 'body_strikes',
    'leg_strike_atts', 'leg_strikes', 'dist_strike_atts', 'dist_strikes',
    'clinch_atts', 'clinchs', 'ground_atts', 'grounds', 'total_fight_time',
]

cols_to_drop = ['url_link', 'event_url', 'fights_url']
fights_with_event_processed = fights_with_event.drop(columns=cols_to_drop, errors='ignore')
fights_with_event_processed = fights_with_event_processed.sort_values('event_date', ascending=True)

In [25]:
# Make per-fighter cumulative stats so each row shows totals *before* that fight

for side in ['f1', 'f2']:
    fighter_id_col = f'{side}_id'
    for metric in STAT_COLS:
        col = f'{side}_{metric}'
        if col in fights_with_event_processed.columns:
            fights_with_event_processed[col] = (
                fights_with_event_processed
                .groupby(fighter_id_col)[col]
                .transform(lambda s: s.fillna(0).cumsum().shift(fill_value=0))
            )

In [26]:
# Experience and win/loss/draw records are computed in the next cell (experience = wins + losses + draws).

In [27]:
# Add per-fighter prior record (wins/draws/losses before each fight)

# fights_with_event_processed is already sorted by event_date
df = fights_with_event_processed.reset_index(drop=True).reset_index().rename(columns={"index": "fight_index"})

# Long format: one row per fighter per fight
rows_f1 = df[["fight_index", "event_date", "event_id", "result", "f1_id", "f2_id"]].rename(
    columns={"f1_id": "fighter_id", "f2_id": "opponent_id"}
)
rows_f1["side"] = "f1"

rows_f2 = df[["fight_index", "event_date", "event_id", "result", "f1_id", "f2_id"]].rename(
    columns={"f2_id": "fighter_id", "f1_id": "opponent_id"}
)
rows_f2["side"] = "f2"

long = pd.concat([rows_f1, rows_f2], ignore_index=True)

# Ensure result and IDs are comparable (e.g. all numeric); avoid string vs int mismatch
long["result"] = pd.to_numeric(long["result"], errors="coerce")
long["fighter_id"] = pd.to_numeric(long["fighter_id"], errors="coerce")
long["opponent_id"] = pd.to_numeric(long["opponent_id"], errors="coerce")

# Outcome flags from the fighter's perspective
long["win_flag"] = (long["result"] == long["fighter_id"]).astype(int)
long["loss_flag"] = (long["result"] == long["opponent_id"]).astype(int)
long["draw_flag"] = (
    ~((long["result"] == long["fighter_id"]) | (long["result"] == long["opponent_id"]))
).astype(int)

# Sort by fighter and time; cumulative record *before* each fight (shift within each fighter)
long = long.sort_values(["fighter_id", "event_date", "event_id", "fight_index", "side"]).reset_index(drop=True)
# Shift must be within each fighter's group so first fight gets 0, not a global shift
long["prior_wins"] = long.groupby("fighter_id")["win_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))
long["prior_losses"] = long.groupby("fighter_id")["loss_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))
long["prior_draws"] = long.groupby("fighter_id")["draw_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))

# Map back to wide table
f1_records = (
    long[long["side"] == "f1"][["fight_index", "prior_wins", "prior_losses", "prior_draws"]]
    .rename(columns={"prior_wins": "f1_wins", "prior_losses": "f1_losses", "prior_draws": "f1_draws"})
)
f2_records = (
    long[long["side"] == "f2"][["fight_index", "prior_wins", "prior_losses", "prior_draws"]]
    .rename(columns={"prior_wins": "f2_wins", "prior_losses": "f2_losses", "prior_draws": "f2_draws"})
)

fights_with_event_processed = (
    df.merge(f1_records, on="fight_index", how="left")
    .merge(f2_records, on="fight_index", how="left")
    .drop(columns=["fight_index"])
)

# Experience = total prior fights = wins + losses + draws (overwrite cumcount from earlier)
fights_with_event_processed["f1_experience"] = (
    fights_with_event_processed["f1_wins"] + fights_with_event_processed["f1_losses"] + fights_with_event_processed["f1_draws"]
)
fights_with_event_processed["f2_experience"] = (
    fights_with_event_processed["f2_wins"] + fights_with_event_processed["f2_losses"] + fights_with_event_processed["f2_draws"]
)

In [28]:
# Convert cumulative stats into per-fight averages (over prior fights)

for side in ['f1', 'f2']:
    exp_col = f'{side}_experience'
    exp = fights_with_event_processed[exp_col].replace(0, pd.NA)
    for metric in STAT_COLS:
        col = f'{side}_{metric}'
        if col in fights_with_event_processed.columns:
            avg_col = fights_with_event_processed[col] / exp
            fights_with_event_processed[col] = avg_col.fillna(0).astype(float)

In [29]:
# (trainable_data.csv is built in the next cell when we write the model base)

In [30]:
# Records and experience are computed in cell 6; model base and trainable are written in the next cell.

In [31]:
# Build modeling-ready CSV with selected columns

# Start from the processed, sorted, URL-free fights data
base = fights_with_event_processed.copy()

selected_cols = [
    'f1_id', 'f2_id',
    'f1_name', 'f2_name',
    'weight_class',
    'f1_age_during', 'f2_age_during',
    'f1_height_cm', 'f2_height_cm',
    'f1_knockdowns', 'f2_knockdowns',
    'f1_sig_strike_atts', 'f2_sig_strike_atts',
    'f1_sig_strikes', 'f2_sig_strikes',
    'f1_tot_strike_atts', 'f2_tot_strike_atts',
    'f1_tot_strikes', 'f2_tot_strikes',
    'f1_takedown_atts', 'f2_takedown_atts',
    'f1_takedowns', 'f2_takedowns',
    'f1_submissions', 'f2_submissions',
    'f1_reversals', 'f2_reversals',
    'f1_ctrl_time', 'f2_ctrl_time',
    'f1_head_strike_atts', 'f2_head_strike_atts',
    'f1_head_strikes', 'f2_head_strikes',
    'f1_body_strike_atts', 'f2_body_strike_atts',
    'f1_body_strikes', 'f2_body_strikes',
    'f1_leg_strike_atts', 'f2_leg_strike_atts',
    'f1_leg_strikes', 'f2_leg_strikes',
    'f1_dist_strike_atts', 'f2_dist_strike_atts',
    'f1_dist_strikes', 'f2_dist_strikes',
    'f1_clinch_atts', 'f2_clinch_atts',
    'f1_clinchs', 'f2_clinchs',
    'f1_ground_atts', 'f2_ground_atts',
    'f1_grounds', 'f2_grounds',
    'f1_total_fight_time', 'f2_total_fight_time',
    'f1_experience', 'f2_experience',
    'f1_wins', 'f1_losses', 'f1_draws', 'f2_wins', 'f2_losses', 'f2_draws',
    'result', 'winner', 'mens_class_encoded', 'womens_class_encoded', 'is_title_bout',
    'event_id', 'event_name', 'event_date',
]

model_df = base[selected_cols]



In [32]:
model_df['f1_experience'].value_counts()

f1_experience
1     932
2     863
0     829
3     682
4     559
5     486
6     454
7     385
8     349
9     323
10    243
11    220
12    218
13    176
14    154
15    138
16    116
17     94
18     70
19     60
21     52
20     49
22     46
23     32
24     24
26     20
25     19
27     18
28     15
29      9
31      9
30      8
32      7
33      4
35      4
34      3
36      2
37      2
38      2
39      2
40      2
41      1
Name: count, dtype: int64

to do before training model:
* *talk about:* drop fights with players with less than 2/3 experience
* make differentials
* make duplicates

# fights_trainable IS NOT TRAINABLE YET

In [33]:
fights_trainable_path = DATA_PROCESSED / 'fights_trainable.csv'

model_df.to_csv(fights_trainable_path, index=False)
print(model_df.shape)

fights_trainable_path

(7681, 71)


WindowsPath('C:/Users/maxle/Documents/ML-Project-clone/data/processed/fights_trainable.csv')